In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers datasets accelerate -q

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import numpy as np

In [ ]:
# -------------------------------
# 1. Load Pre-split Data
# -------------------------------
train_df = pd.read_csv("/content/drive/MyDrive/Data-Single/train_balanced.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Data-Single/test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


In [ ]:
# -------------------------------
# 2. Tokenization
# -------------------------------
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def preprocess(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# -------------------------------
# 3. Model
# -------------------------------
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

# -------------------------------
# 4. Training Arguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./results_roberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs_roberta",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

In [ ]:
# -------------------------------
# 5. Metrics
# -------------------------------
accuracy = load_metric("accuracy")
f1 = load_metric("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy.compute(predictions=preds, references=labels)
    f1_score = f1.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1_score["f1"]}

# -------------------------------
# 6. Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
# -------------------------------
# 8. Evaluate Model
# -------------------------------
results = trainer.evaluate()
print("Evaluation Results:", results)


In [ ]:
# -------------------------------
# 7. Train Model
# -------------------------------
trainer.train()


In [ ]:
# -------------------------------
# 9. Save Model
# -------------------------------
trainer.save_model("/content/drive/MyDrive/Data-Single/roberta_model/")
tokenizer.save_pretrained("/content/drive/MyDrive/Data-Single/roberta_modelPre/")

In [ ]:
# -------------------------------
# 10. Inference Example
# -------------------------------
inference_model = RobertaForSequenceClassification.from_pretrained("/content/drive/MyDrive/Data-Single/roberta_modelPre/")
inference_tokenizer = RobertaTokenizer.from_pretrained("/content/drive/MyDrive/Data-Single/roberta_modelPre/")

In [ ]:
def predict(text):
    inputs = inference_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = inference_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs).item()
    return {"prediction": pred, "confidence": probs[0][pred].item()}
# Example usage
print(predict("Breaking news: Scientists discover water on Mars!"))
print(predict("Celebrity scandal: Famous actor involved in fake controversy."))